# Info sobre outputs

Este arquivo produz as variáveis:
- **roads**  --> gdf de vias com geometria e valores de ADT
- **industrial_gdf**  --> gdf de industrias
- **stations** --> gdf de estações de monitoramento com geometria e poluentes

# 1.0 Importando pacotes e funções

In [19]:
# Pacotes e funções
import geopandas as gpd
from pathlib import Path
from long_2_utm_zone import long_2_utm_zone
from utm_zone_2_epsg import utm_zone_2_epsg
import pandas as pd
import numpy as np
import os

# Desativando notação científica
pd.set_option('display.float_format', '{:.2f}'.format)

# 2.0 Definindo caminhos

In [20]:
# Caminhos
# Caminhos de diretórios
root_path = os.path.dirname(os.getcwd())

inputs_path = root_path + '/data/rep_espacial/inputs'

outputs_path = root_path + '/data/rep_espacial/outputs'

# Caminhos de inputs
roads_path = inputs_path + '/processed_roads_dissolved.parquet'

flow_path = inputs_path + '/prelim_roads.csv'

industrial_path = inputs_path + '/industrial_sites_20250902.gpkg'

stations_path = root_path + '/data/Monitoramento_QAr_BR.csv' # constantemente atualizado

# 3.0 Geodataframes de Vias do Brasil

## 3.1 Lendo arquivo do modelo estático de vias

In [21]:
"""Lendo geodataframe de vias, com as colunas obrigatórias:
    'osm_id': int de código de identificação de vias do OpenStreetMaps
    'geometry': LineString de geometria da via
"""
# Lendo geodataframe
roads = gpd.read_parquet(path=roads_path).astype({'osm_id': int})

## 3.2 Lendo os dados de fluxo de veículos e cálculo de ADT

In [23]:
# Deixando nomes de colunas variáveis
adt_col = 'average_daily_vehicle_count'
vehicle_count_col = 'vehicle_count'

# Lendo o csv com contagem de veículos
roads_with_adt = pd.read_csv(flow_path)

# Cálculo do ADT para cada código de via
roads_with_adt = roads_with_adt.groupby(['osm_id','weekday'])['vehicle_count'].sum()
roads_with_adt = roads_with_adt.groupby('osm_id').mean().reset_index()

# Renomeando coluna vehicle count para ADT
roads_with_adt = roads_with_adt.rename({vehicle_count_col : adt_col}, axis=1)
roads_with_adt

,osm_id,average_daily_vehicle_count
0,8159668.00,57553.82
1,8504851.00,162042.68
2,10064602.00,3462.68
3,10064611.00,19299.17
4,10064669.00,16021.77
...,...,...
12310,1380024292.00,9110.48
12311,1380251668.00,469.97
12312,1384353139.00,12420.15
12313,1385687375.00,6738.12


## 3.3 Selecionando vias com ADT calculado e > 1000 veículos

In [15]:
# Selecionando vias com ADT calculado
roads = pd.merge(roads, roads_with_adt, how='inner', on='osm_id')

"""Definiu-se 1000 veículos/dia como o fluxo diário médio (ADT) mínimo para uma 
via ser considerada como via principal, termo utilizado no Guia de Monitoramento
da Qualidade do Ar do Brasil."""
# Pegando vias com ADT superior a 1000 veículos/dia
roads = roads[roads[adt_col] > 1000]


# 4.0 Zonas Industriais

## 4.1 Lendo o arquivo de indústrias

In [24]:
""" Esta seção lê o arquivo de indústrias, com as colunas obrigatórias:
        'Razão Social': str.
            Nome comercial do empreendimento
        'geometry': Point ou Polygon.
            Geometrias representando indústrias, áreas de mineração e aterros sanitários
"""
# Lendo arquivo de indústrias
industrial_gdf = gpd.read_file(industrial_path)

# Duplicando a coluna de geometria para transmitir ela após o sjoin_nearest na seção 6.2
industrial_gdf['industry_geom'] = industrial_gdf.geometry


# 5.0 Estações de monitoramento da qualidade do ar

## 5.1 Lendo arquivo, determinando código EPSG e filtragem de poluentes de interesse

Esta seção faz a leitura do arquivo de estações de monitoramento da qualidade do ar 
do Brasil, com as colunas obrigatórias:
- 'LONGITUDE': float.
- 'LATITUDE': float.
- 'COD_POLUENTE': float
- 'POLUENTE': str.
- 'ID_OEMA': str.

Em seguida, cada estação é enquadrada dentro de uma zona UTM e atribui-se o código EPSG correspondente, de acordo com a zona UTM e a latitude de cada uma.

Por fim, dentre todas as linhas de estações, o geodataframe é reduzido às que monitoram os poluentes a seguir:
- monóxido de carbono (CO)
- dióxido de enxofre (SO2)
- dióxido de nitrogênio (NO2)
- ozônio (O3)
- material particulado de diâmetro inferior a 10 micrômetros (MP10)
- material particulado de diâmetro inferior a 2.5 micrômetros (MP2.5)
- material particulado total (PTS)

In [25]:
# Lendo o arquivo de estações de monitoramento
stations = pd.read_csv(filepath_or_buffer=stations_path,
                      dtype={'LONGITUDE': float,
                             'LATITUDE':float,
                             'COD_POLUENTE':float,
                             'POLUENTE':str,
                             'ID_OEMA':str
                            }
                      )

# Transformando em GeoDataFrame
stations = gpd.GeoDataFrame(stations,
                            geometry=gpd.points_from_xy(stations.LONGITUDE,
                                                        stations.LATITUDE,
                                                        crs='EPSG:4326'))
# Determinando a zona UTM para cada estação
stations.loc[:,'utm_zone'] = long_2_utm_zone(stations
                                             .geometry
                                             .centroid
                                             .x)

# Determinando do código EPSG para cada estação
stations.loc[:,'EPSG'] = utm_zone_2_epsg(stations['utm_zone'],
                                         stations.geometry
                                         .centroid
                                         .x)

# Removendo a coluna auxiliar de zona UTM
stations.drop(columns='utm_zone', inplace=True)

# Filtrando as estações que monitoram CO, SO2, O3, NO2, PM10, PM2.5 e PTS
stations = stations[stations['COD_POLUENTE'].isin([1.0, 2.0, 3.0, 4.0,
                                                   5.0, 7.0, 8.0])]


/tmp/ipykernel_495318/1230298206.py:19: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .centroid
/tmp/ipykernel_495318/1230298206.py:25: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  .centroid


In [26]:
stations.columns

Index(['UF', 'CIDADE', 'CD_MUN', 'ID_OEMA', 'ID_MMA', 'ID_MMA_COMPLETO',
       'PROPRIETARIO', 'PROP_ENTIDADE', 'OPERADOR', 'OP_ENTIDADE',
       'FUNCIONAMENTO', 'CATEGORIA', 'METODO', 'CALIBRACAO', 'MARCA', 'MODELO',
       'POLUENTE', 'COD_POLUENTE', 'MOBILIDADE', 'REP_ESPACIAL', 'FINALIDADE',
       'STATUS', 'INICIO', 'FIM', 'LATITUDE', 'LONGITUDE', 'MONITORAR',
       'FONTE', 'CERTIFICACAO', 'COD_UF_IBGE', 'ANOS_MONITORADOS',
       'BASE_DADOS', 'ELEVACAO', 'geometry', 'EPSG'],
      dtype='object')